# 06 — Softmax (Numerically Stable Row-wise Softmax)

## Background

Softmax is the core component of Transformer attention and the standard output layer for classification. GPU implementation challenges:

1. **Numerical stability**: computing $e^x$ directly overflows (float16 max ≈ 65504). The standard fix is to subtract the row maximum: $\text{softmax}(x_j) = e^{x_j - \max_k x_k} / \sum_k e^{x_k - \max_k x_k}$
2. **Multi-pass scan**: the numerically stable version requires three passes — find the max, compute exp and sum, then normalise.
3. **FlashAttention's core idea** (Online Softmax) merges these passes into a streaming-friendly form.

## Problem Definition

Compute row-wise numerically stable softmax on matrix `A [N, M]`:

$$\text{MAX}[i] = \max_j A[i, j]$$
$$B[i,j] = \frac{e^{A[i,j] - \text{MAX}[i]}}{\sum_k e^{A[i,k] - \text{MAX}[i]}}$$

**Three-pass algorithm**:
- **Pass 1**: `MAX[i] = max(A[i, :])`
- **Pass 2**: `B[i,j] = exp(A[i,j] - MAX[i])`, `SUM[i] = sum(B[i,:])`
- **Pass 3**: `B[i,j] /= SUM[i]`

In [ ]:
import tilelang
import tilelang.language as T
import triton
import triton.language as tl
import torch

tilelang.disable_cache()
print("TileLang:", tilelang.__version__)
print("Triton:  ", triton.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A")

## PyTorch Reference

In [ ]:
N, M = 4096, 16384
A = torch.randn(N, M, dtype=torch.float32, device="cuda")

def ref_softmax(A):
    return torch.softmax(A, dim=1)

B_ref = ref_softmax(A)
print(f"A: {A.shape} → B: {B_ref.shape}")
print(f"Row-sum check (should ≈ 1): B[0].sum() = {B_ref[0].sum().item():.6f}")

## Triton Implementation

One Triton program per row; three-pass scan via Python loop. Key details:
- `tl.max(chunk, 0)` reduces a `[BLOCK_M]` tensor to a scalar (0-d Triton tensor)
- `tl.maximum(scalar_a, scalar_b)` updates the running row maximum
- Scalar `row_sum` broadcasts over `[BLOCK_M]` in `y / row_sum`

In [ ]:
@triton.jit
def _softmax_kernel(X_ptr, Y_ptr, M, BLOCK_M: tl.constexpr):
    row  = tl.program_id(0)
    base = X_ptr + row * M

    # Pass 1: find row max
    row_max = tl.load(base).to(tl.float32)   # initialise with first element
    for start in range(0, M, BLOCK_M):
        cols  = start + tl.arange(0, BLOCK_M)
        chunk = tl.load(base + cols, mask=cols < M, other=float("-inf")).to(tl.float32)
        row_max = tl.maximum(row_max, tl.max(chunk, 0))

    # Pass 2: exp(x - max), write out, accumulate sum
    row_sum = 0.0
    for start in range(0, M, BLOCK_M):
        cols = start + tl.arange(0, BLOCK_M)
        mask = cols < M
        x = tl.load(base + cols, mask=mask, other=0.0).to(tl.float32)
        e = tl.exp(x - row_max)
        tl.store(Y_ptr + row * M + cols, e, mask=mask)
        row_sum = row_sum + tl.sum(e, 0)

    # Pass 3: normalise
    for start in range(0, M, BLOCK_M):
        cols = start + tl.arange(0, BLOCK_M)
        mask = cols < M
        y = tl.load(Y_ptr + row * M + cols, mask=mask)
        tl.store(Y_ptr + row * M + cols, y / row_sum, mask=mask)

BLOCK_M_TRI = 1024

def triton_softmax(A):
    B = torch.empty_like(A)
    _softmax_kernel[(A.shape[0],)](A, B, A.shape[1], BLOCK_M=BLOCK_M_TRI)
    return B

B_tri = triton_softmax(A)
ok = torch.allclose(B_ref, B_tri, atol=1e-4)
print("Triton correctness:", "✅ PASSED" if ok else
      f"❌ FAILED  max_diff={torch.max(torch.abs(B_ref - B_tri)).item():.6f}")

## TileLang Implementation

Same three-pass algorithm: `T.fill(mx, -inf)` initialises, `T.reduce_max` finds the row maximum, `T.exp2` with `log2_e` implements fast exp, `T.reduce_sum` accumulates.

In [ ]:
# Optimised for gfx1100: 1 row per block (matches Triton's strategy).
# Each block handles a single row sequentially in BLOCK_M-wide tiles,
# maximising occupancy (4096 concurrent blocks) and reducing serial iterations.
BN, BM = 1, 512     # 1 row per block, 512-wide tile

@tilelang.jit(pass_configs={
    tilelang.PassConfigKey.TL_DISABLE_WARP_SPECIALIZED: True,
    tilelang.PassConfigKey.TL_DISABLE_TMA_LOWER: True,
})
def tl_softmax(A, BLOCK_M: int):
    log2_e = 1.44269504
    N_, M_ = T.const("N, M")
    dtype = T.float32
    A: T.Tensor((N_, M_), dtype)
    B = T.empty((N_, M_), dtype)
    with T.Kernel(N_, threads=256) as row:
        A_l = T.alloc_fragment((1, BLOCK_M), dtype)
        B_l = T.alloc_fragment((1, BLOCK_M), dtype)
        mx  = T.alloc_fragment((1,), dtype)
        sm  = T.alloc_fragment((1,), dtype)
        T.fill(mx, float("-inf"))
        for m_blk in T.Serial(M_ // BLOCK_M):
            T.copy(A[row, m_blk * BLOCK_M], A_l)
            T.reduce_max(A_l, mx, dim=1, clear=False)
        T.clear(sm)
        for m_blk in T.Serial(M_ // BLOCK_M):
            T.copy(A[row, m_blk * BLOCK_M], A_l)
            for j in T.Parallel(BLOCK_M):
                B_l[0, j] = T.exp2(log2_e * (A_l[0, j] - mx[0]))
            T.reduce_sum(B_l, sm, dim=1, clear=False)
            T.copy(B_l, B[row, m_blk * BLOCK_M])
        for m_blk in T.Serial(M_ // BLOCK_M):
            T.copy(B[row, m_blk * BLOCK_M], B_l)
            for j in T.Parallel(BLOCK_M):
                B_l[0, j] = B_l[0, j] / sm[0]
            T.copy(B_l, B[row, m_blk * BLOCK_M])
    return B

k = tl_softmax.compile(N=N, M=M, BLOCK_M=BM)
B_tl = k(A.clone())
# atol=5e-4: exp2(x*log2_e) approximation has ~2e-4 max error in float32
ok = torch.allclose(B_ref, B_tl, atol=5e-4)
print("TileLang correctness:", "\u2705 PASSED" if ok else
      f"\u274c FAILED  max_diff={torch.max(torch.abs(B_ref - B_tl)).item():.6f}")


## Performance Comparison

In [ ]:
import matplotlib.pyplot as plt

WARMUP, REPEAT = 20, 200

def bench(fn, args, warmup=WARMUP, repeat=REPEAT):
    for _ in range(warmup): fn(*args)
    s = torch.cuda.Event(enable_timing=True)
    e = torch.cuda.Event(enable_timing=True)
    torch.cuda.synchronize(); s.record()
    for _ in range(repeat): fn(*args)
    e.record(); torch.cuda.synchronize()
    return s.elapsed_time(e) / repeat

results = {
    "PyTorch":  bench(ref_softmax,    [A]),
    "Triton":   bench(triton_softmax, [A]),
    "TileLang": bench(k,             [A]),
}

bytes_rw = N * M * 4 * 2
for name, ms in results.items():
    print(f"  {name:10s}: {ms:.4f} ms  ({bytes_rw / ms / 1e9:.2f} TB/s)")

colors = ["#4C72B0", "#55A868", "#C44E52"]
labels = list(results.keys())
values = list(results.values())

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

ax = axes[0]
bars = ax.bar(labels, values, color=colors, width=0.45, edgecolor="white", linewidth=0.8)
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(values)*0.01,
            f"{val:.4f} ms", ha="center", va="bottom", fontsize=10)
ax.set_ylabel("Latency (ms)"); ax.set_title(f"Softmax Latency  (N={N}, M={M}, float32)")
ax.set_ylim(0, max(values)*1.25); ax.spines[["top","right"]].set_visible(False)

ax = axes[1]
bws = [bytes_rw / ms / 1e9 for ms in values]
bars = ax.bar(labels, bws, color=colors, width=0.45, edgecolor="white", linewidth=0.8)
for bar, val in zip(bars, bws):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(bws)*0.01,
            f"{val:.2f}", ha="center", va="bottom", fontsize=10)
ax.set_ylabel("Effective Bandwidth (TB/s)"); ax.set_title(f"Softmax Bandwidth  (N={N}, M={M}, float32)")
ax.set_ylim(0, max(bws)*1.25); ax.spines[["top","right"]].set_visible(False)

plt.tight_layout()
plt.show()